### ЗАДАЧА: Реестр товаров 

В этой задаче нужно собрать простой каталог товаров для внутренней системы закупок.
Данные приходят как строки, часть строк может быть некорректной.

НЕОБХОДИМО РЕАЛИЗОВАТЬ:

1. Класс `Product`:
   - поля `id`, `name`, `_price`, `category`
   - `@property price` (только чтение)
   - метод `set_price(value) -> bool`:
     - возвращает `True`, если цена валидна и обновлена
     - возвращает `False`, если цена невалидна
   - `@classmethod from_line(cls, raw)`:
     - если строка корректная, вернуть `(product, "")`
     - если ошибка, вернуть `(None, "текст причины")`.

2. Класс `ProductRegistry`:
   - хранит список товаров
   - метод `add(product) -> (bool, str)`:
     - `(True, "")`, если успешно
     - `(False, "...")`, если `id` уже существует
   - метод `count()`
   - метод `all_products()`
   - метод `has_id(id)`
   - метод `by_category()` -> `{category: count}`
   - метод `avg_price()`.

3. Загрузка данных:
   - пройти по `lines`
   - использовать `Product.from_line(...)`
   - если товар создан, пытаться `registry.add(...)`
   - все проблемные строки складывать в `problems` как `(line, reason)`.

4. Вывод:
   - число валидных товаров
   - статистика по категориям
   - средняя цена
   - список проблемных строк.


- Для проверки дублей хранить set из `id`.
- В `avg_price()` вернуть `0`, если товаров нет.
- Для проверки, что строка число, можно использовать вспомогательную функцию с `str.replace('.', '', 1).isdigit()`.


In [ ]:
lines = [
    "P-1;Mouse;25;Periphery",
    "P-2;Keyboard;45;Periphery",
    "P-3;Cloud VM;120;Services",
    "P-2;Duplicate;50;Periphery",
    "P-4;Broken;-5;Services",
    "P-5;BadPrice;abc;Services",
]


def is_number(text):
    # TODO: вернуть True, если text можно считать числом (включая дроби и знак), иначе False
    text = str(text)
    if text == "":
        return False
    
    if len(text) > 0 and text[0] == "-":
        text_without_minus = text[1:]
    else:
        text_without_minus = text

    dot_count = text_without_minus.count(".")

    if dot_count > 1:
        return False
    
    digits_part = text_without_minus.replace(".", "")

    if digits_part.isdigit():
        return True
    else:
        return False


class Product:
    # TODO: __init__(id, name, price, category)
    # TODO: property price (только getter)
    # TODO: set_price(value) -> bool
    # TODO: classmethod from_line(cls, raw) -> (product_or_none, reason)
    def __init__(self, id, name, prcie, category):
        self.id = id
        self.name = name
        self._price = prcie
        self.category = category

    @property
    def price(self):
        return self._price
    
    def set_price(self, value):
        if is_number(value) and float(value) >= 0:
            self._price = float(value)
            return True
        else:
            return False
        
    @classmethod
    def from_line(cls, raw):
        parts = raw.split(";")

        if len(parts) != 4:
            return None, "Неверный формат: должно быть 4 поля"
        
        product_id, name, price_str, category = parts

        if not is_number(price_str):
            return None, f"Неверная цена: '{price_str}' не является числом"
        
        price = float(price_str)

        if price < 0:
            return None, f"Отрицательная цена: {price}"
        
        product = cls(product_id, name, price, category)
        return product, None


class ProductRegistry:
    # TODO: __init__ (products list + id set)
    # TODO: add(product) -> (bool, reason)
    # TODO: count()
    # TODO: all_products()
    # TODO: has_id(id)
    # TODO: by_category()
    # TODO: avg_price()
    def __init__(self):
        self._products = []
        self._ids = set()

    def add(self, product):
        if product.id in self._ids:
            return False, f"Дубликат ID: {product.id}"
        self._products.append(product)
        self._ids.add(product.id)
        return True, None
    
    def count(self):
        return len(self._products)
    
    def all_products(self):
        return self._products
    
    def has_id(self, id):
        return id in self._ids
    
    def by_category(self):
        categories = {}
        for product in self._products:
            category = product.category
            if category not in categories:
                categories[category] = []
            categories[category].append(product)
        return categories
    

    def avg_price(self):
        if not self._products:
            return 0.0
        total = sum(product.price for product in self._products)
        return total / len(self._products)

# TODO: создать registry и problems
# TODO: пройти по lines, загрузить валидные товары
# TODO: проблемные строки добавлять в problems как (line, reason)
# TODO: вывести count, by_category, avg_price и problems

registry = ProductRegistry()
problems = []

for line in lines:
    product, reason = Product.from_line(line)

    if product is not None:
        success, add_reason = registry.add(product)
        if not success:
            problems.append((line, add_reason))
    else:
        problems.append((line, reason))

print("Количество товаров:", registry.count())
print("\nТовары по категориям:")
for category, products in registry.by_category().items():
    print(f"{category}: {len:(products)} товаров")
    for p in products:
        print(f"{p.name} (${p.price})")

print(f"\nСредняя цена: $", registry.avg_price())
print("\nПроблемы при обработке:")
for line, reason in problems:
    print(f"'{line} {reason}")
